In [ ]:
import georinex as gr
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import gnss_tools

#### Calculating TEC

Calculating TEC depends on the frequency of the carrier wavelengths. In this example we will work though some basic TEC calculation.

In [ ]:
# Constants
c = 299792458.0 # m/s
f1 = 1.57542e9  # Hz
f2 = 1.22760e9  # Hz
f5 = 1.17645e9  # Hz


Our station (SMST) does not have coordinates in the RINEX header, but I have the ECEF coords here:

In [ ]:
stn_coords = [-3822374.523, 3699361.161, 3507586.837]

In [ ]:
file = r"data/smst0810.26d.gz"
obs = gr.load(file, use=['G','E'], fast=True)

In [ ]:
fig, axs = plt.subplots(2,1)

axs[0].plot(obs['time'], obs['L1'])
axs[0].plot(obs['time'], obs['C1'])

axs[1].plot(obs['time'], obs['L2'])
axs[1].plot(obs['time'], obs['P2'])


We have to convert from $\phi_i$ to $\Phi_i$

Recall that $\Phi_i = \lambda_i \phi_i$

In [ ]:
obs['L1'] = c * obs['L1'] / f1
obs['L2'] = c * obs['L2'] / f2

In [ ]:
fig, axs = plt.subplots(2,1)

fig.tight_layout()

axs[0].plot(obs['time'], obs['L1'])
axs[0].plot(obs['time'], obs['C1'])

axs[1].plot(obs['time'], obs['L2'])
axs[1].plot(obs['time'], obs['P2'])

In [ ]:
# Convert from 
fig, axs = plt.subplots(2,1)
fig.tight_layout()

axs[0].plot(obs['time'], obs['L1'] - obs['L2'])

axs[1].plot(obs['time'], (obs['P2'] - obs['C1']))


In [ ]:
# proportionality factor
tecScale = ((f2**2) * (f1**2) / (40.3 * (f1**2 - f2**2))) * 1e-16 # convert to TECU

# phase tec
TEC_phase = tecScale * (obs['L1'] - obs['L2'])

# Code TEC
TEC_code = tecScale * (obs['P2'] - obs['C1'])

In [ ]:
# Convert from 
fig, axs = plt.subplots(2,1)
fig.tight_layout()

axs[0].plot(obs['time'], TEC_phase)

axs[1].plot(obs['time'], TEC_code)

In [ ]:
# make subplots
fig, axs = plt.subplots(2,2, squeeze=False)
fig.tight_layout()

for i, sv in enumerate(obs['sv']):
    
    # make new plot if current one is full
    j, k = divmod(i, axs.size)
    if i > 0 and k == 0:
        fig, axs = plt.subplots(*axs.shape)
        fig.tight_layout()
        
    ax = axs[np.unravel_index(k, axs.shape)]

    # find mean difference between phase and code TEC
    arc_diff = np.mean((TEC_code - TEC_phase).sel(sv=sv))

    # plot code, phase, and phase-levelled TEC
    ax.set_title(f"TEC {sv.values}")
    ax.plot(obs['time'], TEC_code.sel(sv=sv), label='Code')
    ax.plot(obs['time'], TEC_phase.sel(sv=sv), label='Phase')
    ax.plot(obs['time'], TEC_phase.sel(sv=sv) + arc_diff, label='Levelled')
    ax.legend()

If data is missing from the RINEX file, a cycle slip may happen without us knowing. 

In [ ]:
# detect if any missing times in the data
time_skip = np.atleast_2d(
        np.hstack((obs.time.diff("time") > np.timedelta64(30, "s"), [False]))
    ).T

Any NaN data from the receiver is invalid (inclusing when the satellite is not visible)

In [ ]:
# detect if any nan data in the phase
nan_tec = np.isnan(TEC_code + TEC_phase)

If the derivative (TECU/s) is too large it may indicate a cycle slip.

In [ ]:
# detect if any TEC derivatives are too big
deriv_tec = np.abs((TEC_phase - TEC_code).differentiate(coord='time', datetime_unit='s')) > 1 # TECU

If the jump from one time to another is too large it may also indicate a cycle slip.

In [ ]:
# detect if any jumps in TEC
jump_tec = np.abs(np.diff(TEC_phase, prepend=TEC_phase.isel(time=slice(0,1)), axis=0)) > 10

Any data that fulfills these conditions is flagged as edge of a valid arc.

In [ ]:
# everything that is True is a place with bad TEC (or a cycle slip)
bad_tec = (time_skip + nan_tec + jump_tec + deriv_tec).astype(bool).T

We then need to find all the points where arcs begin and end.

In [ ]:
# find all the start points and end points of good data
(ss_list, ts_list) = np.nonzero(np.diff(bad_tec, prepend=1, axis=1) < 0)
(se_list, te_list) = np.nonzero(np.diff(bad_tec, append=1, axis=1) > 0)

# only keep arcs longer than 5 minutes
igood = (obs.time[te_list].data - obs.time[ts_list].data) > np.timedelta64(
            5, "m"
        )

ts_list = ts_list[igood]
te_list = te_list[igood]
ss_list = ss_list[igood]
se_list = se_list[igood]

Go through each of the arcs and level the phase to the code.

In [ ]:
stec = np.full_like(TEC_code, np.nan)
stec_error = np.full_like(TEC_code, np.nan)
arc_index = np.full(TEC_code.shape, fill_value=-1, dtype=int)

n = 0
for s, ts, te in zip(ss_list, ts_list, te_list):
    n += 1
   
    plo = np.nanmean(TEC_code[ts: te + 1, s] - TEC_phase[ts: te + 1, s]) 

    assert not np.isnan(plo)

    stec[ts: te + 1, s] = TEC_phase[ts: te + 1, s] + plo

    # residuals to estimate error
    nrms = np.sqrt(np.nanmean((stec[ts: te + 1, s] - TEC_code[ts: te + 1, s])**2))
    stec_error[ts: te + 1, s] = nrms/np.sqrt(1 + te-ts)

    arc_index[ts: te + 1, s] = np.short(n)

In [ ]:
# make subplots
fig, axs = plt.subplots(2,2, squeeze=False)
fig.tight_layout()

for i, sv in enumerate(obs['sv']):
    
    # make new plot if current one is full
    j, k = divmod(i, axs.size)
    if i > 0 and k == 0:
        fig, axs = plt.subplots(*axs.shape)
        fig.tight_layout()
        
    ax = axs[np.unravel_index(k, axs.shape)]

    # plot code, phase, and phase-levelled TEC
    ax.set_title(f"TEC {sv.values}")
    ax.plot(obs['time'], TEC_code.sel(sv=sv), label='Code')
    #ax.plot(obs['time'], TEC_phase.sel(sv=sv), label='Phase')
    ax.plot(obs['time'], stec[:, i], 'k', label='Levelled (properly)')
    ax.plot(obs['time'], stec[:, i] + stec_error[:,i], ':k' )
    ax.plot(obs['time'], stec[:, i] - stec_error[:,i], ':k' )
    ax.legend()

This technique has been written up as a function in `gnss_tools.py` for you to use. 

In [ ]:
data = gnss_tools.simple_phase_level(TEC_code, TEC_phase, max_error=4.5)

In [ ]:
data

In [ ]:
# make subplots
fig, axs = plt.subplots(2,2)
fig.tight_layout()

for i, sv in enumerate(obs['sv']):
    
    # make new plot if current one is full
    j, k = divmod(i, axs.size)
    if i > 0 and k == 0:
        fig, axs = plt.subplots(*axs.shape)
        fig.tight_layout()
        
    ax = axs[np.unravel_index(k, axs.shape)]

    # plot code, phase, and phase-levelled TEC
    ax.set_title(f"TEC {sv.values}")
    ax.plot(obs['time'], TEC_code.sel(sv=sv), label='Code')
    ax.plot(obs['time'], TEC_phase.sel(sv=sv), label='Phase')
    ax.plot(obs['time'], data['sTEC'].sel(sv=sv),  label='Levelled (properly)')
    ax.legend()